# BÀI TẬP THỰC HÀNH CHƯƠNG 4
## Phát hiện biên & Phân vùng ảnh

---

## 📁 Cấu trúc thư mục

```
chapter_4_lab/
├── images/              # Ảnh đầu vào
├── output/              # Kết quả xử lý
└── chapter_4_lab.ipynb  # File notebook (đặt trực tiếp ở thư mục ngoài)
```

---

## ⚙️ 0. Chuẩn bị môi trường

**Cell 0 — Khởi tạo chung (chạy đầu tiên trong notebook):**

In [ ]:
%matplotlib inline

import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from skimage import data, segmentation, color, filters, measure
from skimage.segmentation import slic, mark_boundaries
from skimage.filters import threshold_multiotsu
from collections import deque

Path("images").mkdir(exist_ok=True)
Path("output").mkdir(exist_ok=True)

print("OpenCV version:", cv2.__version__)
print("NumPy version :", np.__version__)

# ---------- Hàm tiện ích hiển thị ----------
def show_image(img, title="", cmap=None, figsize=(5, 4), save_path=None):
    plt.figure(figsize=figsize)
    if img.ndim == 3:
        plt.imshow(img)
    else:
        plt.imshow(img, cmap=cmap or 'gray')
    plt.title(title)
    plt.axis('off')
    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.show()


def show_grid(images, titles, ncols=3, figsize=(15, 8),
              cmap='gray', save_path=None, main_title=None):
    n = len(images)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = np.array(axes).ravel()
    for i, (im, t) in enumerate(zip(images, titles)):
        ax = axes[i]
        if im.ndim == 3:
            ax.imshow(im)
        else:
            ax.imshow(im, cmap=cmap)
        ax.set_title(t, fontsize=10)
        ax.axis('off')
    for j in range(n, len(axes)):
        axes[j].axis('off')
    if main_title:
        plt.suptitle(main_title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.show()


# ---------- Chuẩn hóa hiển thị ----------
def norm_vis(x):
    """Chuẩn hóa mảng float về uint8 để hiển thị."""
    x = np.abs(x)
    if x.max() == 0:
        return np.zeros_like(x, dtype=np.uint8)
    return (x / x.max() * 255).astype(np.uint8)

> **Lưu ý:** Không dùng `cv2.imshow` / `cv2.waitKey` để tránh treo kernel.

---

## Bài 1: Phát hiện điểm biệt lập (Point Detection)

**Mức độ:** Cơ bản

**Mục tiêu:** Cài đặt mặt nạ Laplacian 8-láng giềng và phát hiện điểm biệt lập bằng ngưỡng.

**Yêu cầu:**
1. Tạo một ảnh tổng hợp `300×300` nền xám 100, đặt 5–7 điểm sáng đơn lẻ (giá trị 255) ở các vị trí ngẫu nhiên.
2. Định nghĩa mặt nạ Laplacian:
   ```
   [-1 -1 -1]
   [-1  8 -1]
   [-1 -1 -1]
   ```
3. Tích chập ảnh với mặt nạ này (`cv2.filter2D`).
4. Áp dụng ngưỡng `|R| > T` (chọn `T = 200`) để xác định điểm biệt lập.
5. Hiển thị ảnh gốc, đáp ứng Laplacian (chuẩn hóa), và ảnh đã phát hiện điểm.
6. Lưu kết quả vào `output/bai1_point_detect.png`.

**Lời giải:**

In [ ]:
# 1. Tạo ảnh tổng hợp
np.random.seed(42)
img = (np.ones((300, 300)) * 100).astype(np.uint8)

# Đặt các điểm sáng ngẫu nhiên
n_points = 6
positions = []
for _ in range(n_points):
    y, x = np.random.randint(20, 280, 2)
    img[y, x] = 255
    positions.append((y, x))
print("Vị trí các điểm biệt lập:", positions)

# 2. Mặt nạ Laplacian point detection
kernel_point = np.array([
    [-1, -1, -1],
    [-1,  8, -1],
    [-1, -1, -1]
], dtype=np.float32)

# 3. Tích chập
response = cv2.filter2D(img.astype(np.float32), -1, kernel_point)

# 4. Ngưỡng
T = 200
detected = (np.abs(response) > T).astype(np.uint8) * 255

# 5. Hiển thị
show_grid(
    [img, norm_vis(response), detected],
    ['Ảnh gốc (nền 100, điểm 255)',
     'Đáp ứng Laplacian (chuẩn hóa)',
     f'Điểm phát hiện (T={T})'],
    ncols=3, figsize=(15, 5),
    save_path='output/bai1_point_detect.png',
    main_title='Bài 1: Phát hiện điểm biệt lập'
)

# Đếm
n_detected = np.count_nonzero(detected)
print(f"Số pixel được phát hiện: {n_detected} (mỗi điểm có thể chiếm vài pixel)")

**📸 Kết quả:** Các điểm sáng đơn lẻ được phát hiện chính xác; nền phẳng không cho đáp ứng.

---

## Bài 2: Phát hiện đường (Line Detection)

**Mức độ:** Cơ bản

**Mục tiêu:** Dùng 4 mặt nạ định hướng để phát hiện đường ngang, dọc, chéo ±45°.

**Yêu cầu:**
1. Tạo ảnh `300×300` nền đen, vẽ 4 đường trắng mảnh theo 4 hướng: ngang, dọc, chéo +45°, chéo −45°.
2. Định nghĩa 4 mặt nạ:
   - Ngang: `[[-1,-1,-1],[2,2,2],[-1,-1,-1]]`
   - Dọc: `[[-1,2,-1],[-1,2,-1],[-1,2,-1]]`
   - Chéo +45°: `[[-1,-1,2],[-1,2,-1],[2,-1,-1]]`
   - Chéo −45°: `[[2,-1,-1],[-1,2,-1],[-1,-1,2]]`
3. Tích chập từng mặt nạ với ảnh, ngưỡng `T = 200`.
4. Hiển thị 4 kết quả.
5. Lưu kết quả vào `output/bai2_line_detect.png`.

**Lời giải:**

In [ ]:
# 1. Tạo ảnh
img = np.zeros((300, 300), dtype=np.uint8)
cv2.line(img, (50, 75),  (250, 75),  255, 1)   # ngang
cv2.line(img, (75, 50),  (75, 250),  255, 1)   # dọc
cv2.line(img, (50, 50),  (250, 250), 255, 1)   # chéo +45
cv2.line(img, (250, 50), (50, 250),  255, 1)   # chéo -45

# 2. Bốn mặt nạ định hướng
masks = {
    'Ngang':    np.array([[-1,-1,-1],[ 2, 2, 2],[-1,-1,-1]], dtype=np.float32),
    'Dọc':      np.array([[-1, 2,-1],[-1, 2,-1],[-1, 2,-1]], dtype=np.float32),
    'Chéo +45': np.array([[-1,-1, 2],[-1, 2,-1],[ 2,-1,-1]], dtype=np.float32),
    'Chéo -45': np.array([[ 2,-1,-1],[-1, 2,-1],[-1,-1, 2]], dtype=np.float32),
}

T = 200
results = []
titles  = []
for name, k in masks.items():
    resp = cv2.filter2D(img.astype(np.float32), -1, k)
    det  = (resp > T).astype(np.uint8) * 255
    results.append(det)
    titles.append(f'{name}\n(T={T})')

show_grid(
    [img] + results,
    ['Ảnh gốc (4 đường)'] + titles,
    ncols=3, figsize=(15, 10),
    save_path='output/bai2_line_detect.png',
    main_title='Bài 2: Phát hiện đường theo 4 hướng'
)

# Nhận xét: mỗi mặt nạ chỉ phản ứng mạnh với đường có hướng tương ứng.

**📸 Kết quả:** Mỗi mặt nạ chỉ "bắt" được đường đúng hướng — các đường khác không xuất hiện hoặc rất mờ.

---

## Bài 3: So sánh toán tử Roberts, Prewitt, Sobel

**Mức độ:** Trung bình

**Mục tiêu:** Cài đặt thủ công 3 toán tử gradient và so sánh kết quả.

**Yêu cầu:**
1. Tải ảnh `data.camera()`.
2. Cài đặt 3 cặp kernel:
   - **Roberts (2×2):** `Gx=[[1,0],[0,-1]]`, `Gy=[[0,1],[-1,0]]`
   - **Prewitt (3×3):** `Gx=[[-1,0,1],[-1,0,1],[-1,0,1]]`, `Gy=[[-1,-1,-1],[0,0,0],[1,1,1]]`
   - **Sobel (3×3):** `Gx=[[-1,0,1],[-2,0,2],[-1,0,1]]`, `Gy=[[-1,-2,-1],[0,0,0],[1,2,1]]`
3. Với mỗi phương pháp, tính `M = sqrt(Gx² + Gy²)` và chuẩn hóa.
4. Hiển thị 3 kết quả cạnh nhau.
5. Lưu kết quả vào `output/bai3_gradient_operators.png`.

**Lời giải:**

In [ ]:
img = data.camera().astype(np.float32)

# 1. Định nghĩa kernel
operators = {
    'Roberts': (
        np.array([[1, 0], [0, -1]], dtype=np.float32),
        np.array([[0, 1], [-1, 0]], dtype=np.float32)
    ),
    'Prewitt': (
        np.array([[-1,0,1],[-1,0,1],[-1,0,1]], dtype=np.float32),
        np.array([[-1,-1,-1],[0,0,0],[1,1,1]], dtype=np.float32)
    ),
    'Sobel': (
        np.array([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=np.float32),
        np.array([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=np.float32)
    ),
}

# 2. Tính magnitude
magnitudes = {}
for name, (Kx, Ky) in operators.items():
    Gx = cv2.filter2D(img, -1, Kx)
    Gy = cv2.filter2D(img, -1, Ky)
    M  = np.sqrt(Gx**2 + Gy**2)
    magnitudes[name] = norm_vis(M)

# 3. Hiển thị
show_grid(
    [img.astype(np.uint8)] + list(magnitudes.values()),
    ['Ảnh gốc'] + list(magnitudes.keys()),
    ncols=4, figsize=(18, 5),
    save_path='output/bai3_gradient_operators.png',
    main_title='Bài 3: So sánh Roberts / Prewitt / Sobel'
)

# NHẬN XÉT:
# - Roberts: nhanh nhưng nhạy nhiễu, biên mảnh.
# - Prewitt: ổn định hơn, biên dày.
# - Sobel: trọng số tâm cao → giảm nhiễu tốt nhất, biên rõ và mượt.

**📸 Kết quả:** Cả 3 đều làm nổi bật biên; Sobel cho kết quả "sạch" và ổn định nhất.

---

## Bài 4: Phát hiện biên Canny

**Mức độ:** Trung bình

**Mục tiêu:** Áp dụng Canny và khảo sát ảnh hưởng của 2 ngưỡng.

**Yêu cầu:**
1. Tải ảnh `data.camera()`.
2. Chạy Canny với các cặp ngưỡng:
   - `(50, 100)`, `(100, 200)`, `(150, 250)`, `(200, 300)`
3. Hiển thị 4 kết quả + ảnh gốc.
4. Nhận xét về ảnh hưởng của ngưỡng.
5. Lưu kết quả vào `output/bai4_canny.png`.

**Lời giải:**

In [ ]:
img = data.camera()

threshold_pairs = [(50, 100), (100, 200), (150, 250), (200, 300)]
canny_imgs = [cv2.Canny(img, t1, t2) for t1, t2 in threshold_pairs]

show_grid(
    [img] + canny_imgs,
    ['Ảnh gốc'] + [f'Canny ({t1}, {t2})' for t1, t2 in threshold_pairs],
    ncols=3, figsize=(15, 10),
    save_path='output/bai4_canny.png',
    main_title='Bài 4: Phát hiện biên Canny với các ngưỡng khác nhau'
)

# NHẬN XÉT:
# - Ngưỡng thấp → nhiều biên (bao gồm cả nhiễu).
# - Ngưỡng cao   → chỉ giữ biên mạnh, nhưng có thể làm đứt biên yếu.
# - Tỷ lệ t1:t2 lý tưởng là 1:2 hoặc 1:3 theo khuyến nghị của Canny.

**📸 Kết quả:** Ngưỡng càng cao → càng ít biên nhưng cũng dễ mất chi tiết.

---

## Bài 5: LoG — Laplacian of Gaussian (Marr-Hildreth)

**Mức độ:** Trung bình

**Mục tiêu:** Cài đặt pipeline LoG: Gaussian → Laplacian → zero-crossing.

**Yêu cầu:**
1. Tải ảnh `data.camera()`.
2. Làm mịn bằng Gaussian `σ=1.5, ksize=9×9`.
3. Tính Laplacian của ảnh đã làm mịn.
4. Tìm zero-crossing: pixel có dấu đổi so với láng giềng.
5. Hiển thị ảnh gốc, ảnh mịn, Laplacian (chuẩn hóa), và ảnh zero-crossing.
6. Lưu kết quả vào `output/bai5_log.png`.

**Lời giải:**

In [ ]:
img = data.camera()

# 1. Làm mịn Gaussian
blurred = cv2.GaussianBlur(img, (9, 9), sigmaX=1.5)

# 2. Laplacian
lap = cv2.Laplacian(blurred.astype(np.float32), cv2.CV_32F)

# 3. Zero-crossing
def zero_crossing(lap):
    """Phát hiện zero-crossing: dấu đổi giữa các láng giềng."""
    h, w = lap.shape
    zc = np.zeros((h, w), dtype=np.uint8)
    sign = np.sign(lap)
    for i in range(1, h-1):
        for j in range(1, w-1):
            patch = sign[i-1:i+2, j-1:j+2]
            if patch.max() > 0 and patch.min() < 0:
                zc[i, j] = 255
    return zc

zc = zero_crossing(lap)

# 4. Hiển thị
show_grid(
    [img, blurred, norm_vis(lap), zc],
    ['Ảnh gốc',
     'Gaussian σ=1.5',
     '|Laplacian| (chuẩn hóa)',
     'Zero-crossing (biên LoG)'],
    ncols=4, figsize=(18, 5),
    save_path='output/bai5_log.png',
    main_title='Bài 5: Marr-Hildreth (LoG) — Gaussian + Laplacian + Zero-crossing'
)

# NHẬN XÉT:
# - Làm mịn trước giúp giảm nhiễu → zero-crossing sạch hơn Laplacian thuần.
# - σ lớn → biên mượt hơn nhưng mất chi tiết nhỏ.

**📸 Kết quả:** LoG cho biên mảnh, sạch; ít nhiễu hơn nhiều so với Laplacian trực tiếp.

---

## Bài 6: Kết hợp Gradient + Phân ngưỡng

**Mức độ:** Trung bình

**Mục tiêu:** Từ gradient Sobel, dùng ngưỡng để tạo ảnh biên nhị phân.

**Yêu cầu:**
1. Tải ảnh `data.camera()`.
2. Tính `M(x, y)` từ Sobel.
3. Chuẩn hóa M về `[0, 255]`.
4. Phân ngưỡng với `T ∈ {30, 60, 100, 150}`.
5. Hiển thị 4 kết quả + M chuẩn hóa; lưu vào `output/bai6_gradient_threshold.png`.

**Lời giải:**

In [ ]:
img = data.camera()

# 1. Tính Sobel magnitude
Gx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
Gy = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
M  = np.sqrt(Gx**2 + Gy**2)
M_norm = np.clip(M / M.max() * 255, 0, 255).astype(np.uint8)

# 2. Phân ngưỡng
thresholds = [30, 60, 100, 150]
edges = [(M_norm > T).astype(np.uint8) * 255 for T in thresholds]

show_grid(
    [img, M_norm] + edges,
    ['Ảnh gốc', 'M chuẩn hóa'] + [f'Biên T={T}' for T in thresholds],
    ncols=3, figsize=(15, 10),
    save_path='output/bai6_gradient_threshold.png',
    main_title='Bài 6: Kết hợp Gradient với phân ngưỡng'
)

**📸 Kết quả:** Ngưỡng càng cao → càng ít pixel biên nhưng chỉ giữ biên mạnh nhất.

---

## Bài 7: Phân ngưỡng toàn cục — Thủ công & Otsu

**Mức độ:** Trung bình

**Mục tiêu:** So sánh ngưỡng thủ công với Otsu tự động.

**Yêu cầu:**
1. Tải ảnh `data.coins()`.
2. Áp dụng `cv2.threshold` với ngưỡng thủ công `T=100`, `T=150`.
3. Áp dụng Otsu (`cv2.THRESH_OTSU`).
4. Hiển thị ảnh gốc + histogram có đánh dấu ngưỡng Otsu + các ảnh nhị phân.
5. Lưu kết quả vào `output/bai7_otsu.png`.

**Lời giải:**

In [ ]:
img = data.coins()

# 1. Ngưỡng thủ công
_, thresh100 = cv2.threshold(img, 100, 255, cv2.THRESH_BINARY)
_, thresh150 = cv2.threshold(img, 150, 255, cv2.THRESH_BINARY)

# 2. Otsu
otsu_val, otsu_img = cv2.threshold(img, 0, 255,
                                    cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print(f"Ngưỡng Otsu tự động tìm được: T = {otsu_val:.2f}")

# 3. Hiển thị
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes[0,0].imshow(img, cmap='gray'); axes[0,0].set_title('Ảnh gốc'); axes[0,0].axis('off')
axes[0,1].imshow(thresh100, cmap='gray'); axes[0,1].set_title('Ngưỡng T=100'); axes[0,1].axis('off')
axes[0,2].imshow(thresh150, cmap='gray'); axes[0,2].set_title('Ngưỡng T=150'); axes[0,2].axis('off')

axes[1,0].imshow(otsu_img, cmap='gray')
axes[1,0].set_title(f'Otsu: T={otsu_val:.1f}'); axes[1,0].axis('off')

axes[1,1].hist(img.ravel(), 256, [0, 256], color='gray')
axes[1,1].axvline(otsu_val, color='red', linestyle='--', label=f'Otsu T={otsu_val:.1f}')
axes[1,1].legend(); axes[1,1].set_title('Histogram + ngưỡng Otsu')

axes[1,2].axis('off')

plt.suptitle('Bài 7: Phân ngưỡng toàn cục — Thủ công vs Otsu',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('output/bai7_otsu.png', dpi=100, bbox_inches='tight')
plt.show()

**📸 Kết quả:** Otsu tự động tìm ngưỡng nằm giữa hai đỉnh histogram → tách nền và đối tượng tốt.

---

## Bài 8: Phân ngưỡng thích nghi (Adaptive Thresholding)

**Mức độ:** Trung bình

**Mục tiêu:** So sánh global vs adaptive trên ảnh có chiếu sáng không đều.

**Yêu cầu:**
1. Tải ảnh `data.page()`.
2. Áp dụng global Otsu + adaptive (`ADAPTIVE_THRESH_GAUSSIAN_C`, `blockSize=15, C=8`).
3. Thử thêm `blockSize=35`.
4. Hiển thị 4 ảnh; lưu vào `output/bai8_adaptive.png`.

**Lời giải:**

In [ ]:
img = data.page()

# 1. Global Otsu
_, global_img = cv2.threshold(img, 0, 255,
                              cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# 2. Adaptive Gaussian
adaptive1 = cv2.adaptiveThreshold(img, 255,
                                   cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY,
                                   blockSize=15, C=8)
adaptive2 = cv2.adaptiveThreshold(img, 255,
                                   cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY,
                                   blockSize=35, C=8)

show_grid(
    [img, global_img, adaptive1, adaptive2],
    ['Ảnh gốc (page)',
     'Global Otsu',
     'Adaptive (blockSize=15, C=8)',
     'Adaptive (blockSize=35, C=8)'],
    ncols=4, figsize=(18, 6),
    save_path='output/bai8_adaptive.png',
    main_title='Bài 8: Phân ngưỡng toàn cục vs thích nghi'
)

# NHẬN XÉT:
# - Global Otsu bị "chết" ở vùng tối (mất chữ).
# - Adaptive xử lý tốt hơn nhờ ngưỡng riêng cho từng vùng cục bộ.
# - blockSize lớn hơn → xử lý vùng rộng, ít nhạy với chi tiết nhỏ.

**📸 Kết quả:** Adaptive cho kết quả vượt trội trên ảnh có nền sáng tối không đều.

---

## Bài 9: Đa ngưỡng (Multi-Otsu)

**Mức độ:** Nâng cao

**Mục tiêu:** Áp dụng Multi-Otsu để phân đoạn ảnh thành nhiều lớp.

**Yêu cầu:**
1. Tải ảnh `data.camera()`.
2. Áp dụng `threshold_multiotsu(img, classes=3)` và `classes=4`.
3. Phân đoạn ảnh theo các ngưỡng tìm được.
4. Hiển thị ảnh gốc + histogram có đánh dấu các ngưỡng + ảnh phân đoạn.
5. Lưu kết quả vào `output/bai9_multiotsu.png`.

**Lời giải:**

In [ ]:
img = data.camera()

# 1. Multi-Otsu
thresh3 = threshold_multiotsu(img, classes=3)
thresh4 = threshold_multiotsu(img, classes=4)
print(f"Ngưỡng 3 lớp: {thresh3}")
print(f"Ngưỡng 4 lớp: {thresh4}")

# 2. Phân đoạn
regions3 = np.digitize(img, bins=thresh3)
regions4 = np.digitize(img, bins=thresh4)

# 3. Hiển thị
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes[0,0].imshow(img, cmap='gray'); axes[0,0].set_title('Ảnh gốc'); axes[0,0].axis('off')
axes[0,1].imshow(regions3, cmap='viridis'); axes[0,1].set_title(f'3 lớp\nT={thresh3}'); axes[0,1].axis('off')
axes[0,2].imshow(regions4, cmap='viridis'); axes[0,2].set_title(f'4 lớp\nT={thresh4}'); axes[0,2].axis('off')

axes[1,0].hist(img.ravel(), 256, [0, 256], color='gray')
for t in thresh3:
    axes[1,0].axvline(t, color='red', linestyle='--')
axes[1,0].set_title('Histogram + ngưỡng 3 lớp')

axes[1,1].hist(img.ravel(), 256, [0, 256], color='gray')
for t in thresh4:
    axes[1,1].axvline(t, color='orange', linestyle='--')
axes[1,1].set_title('Histogram + ngưỡng 4 lớp')

axes[1,2].axis('off')

plt.suptitle('Bài 9: Đa ngưỡng (Multi-Otsu)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('output/bai9_multiotsu.png', dpi=100, bbox_inches='tight')
plt.show()

**📸 Kết quả:** Multi-Otsu chia ảnh thành nhiều lớp (nền tối / trung bình / sáng), phù hợp khi histogram có nhiều đỉnh.

---

## Bài 10: Phát triển vùng (Region Growing)

**Mức độ:** Nâng cao

**Mục tiêu:** Cài đặt Region Growing từ seed point và ngưỡng sai khác.

**Yêu cầu:**
1. Tải ảnh `data.coins()`.
2. Cài đặt `region_growing(img, seed, threshold)`:
   - Dùng BFS (queue) duyệt 8-láng giềng.
   - Thêm pixel vào vùng nếu `|img[p] - seed_value| <= threshold`.
3. Áp dụng với các seed khác nhau và các threshold `{5, 15, 30}`.
4. Hiển thị ảnh gốc + vùng phát triển (mask) trên từng threshold.
5. Lưu kết quả vào `output/bai10_region_growing.png`.

**Lời giải:**

In [ ]:
img = data.coins()

def region_growing(img, seed, threshold=10):
    """Region growing với BFS."""
    h, w = img.shape
    segmented = np.zeros((h, w), dtype=np.uint8)
    seed_value = int(img[seed[0], seed[1]])
    queue = deque([seed])
    segmented[seed[0], seed[1]] = 255

    while queue:
        y, x = queue.popleft()
        for dy in (-1, 0, 1):
            for dx in (-1, 0, 1):
                if dy == 0 and dx == 0:
                    continue
                ny, nx = y + dy, x + dx
                if 0 <= ny < h and 0 <= nx < w and segmented[ny, nx] == 0:
                    if abs(int(img[ny, nx]) - seed_value) <= threshold:
                        segmented[ny, nx] = 255
                        queue.append((ny, nx))
    return segmented

# Chọn seed trên một đồng xu
seed = (150, 180)   # (y, x) — thử điều chỉnh nếu cần

thresholds = [5, 15, 30]
masks = [region_growing(img, seed, T) for T in thresholds]

# Overlay lên ảnh gốc
overlays = []
for m in masks:
    rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    rgb[m > 0] = [255, 100, 100]     # tô đỏ vùng phát triển
    overlays.append(rgb)

show_grid(
    [img] + overlays,
    ['Ảnh gốc'] + [f'Region Growing\nT={T}' for T in thresholds],
    ncols=4, figsize=(18, 5),
    save_path='output/bai10_region_growing.png',
    main_title=f'Bài 10: Region Growing (seed={seed})'
)

# NHẬN XÉT:
# - T nhỏ → vùng nhỏ, chỉ giữ pixel rất giống seed.
# - T lớn → vùng loang ra nhiều, có thể "tràn" sang cả nền.

**📸 Kết quả:** T nhỏ → vùng gọn; T lớn → vùng loang rộng có thể vượt khỏi đối tượng.

---

## Bài 11: Chia tách và Hợp nhất vùng (Split & Merge)

**Mức độ:** Nâng cao

**Mục tiêu:** Cài đặt Split & Merge dùng cây tứ phân (quadtree).

**Yêu cầu:**
1. Tải ảnh `data.camera()`, crop về kích thước `256×256`.
2. Cài đặt:
   - `split(img, x, y, size, threshold)`: nếu phương sai > ngưỡng → chia 4 phần.
   - `merge(blocks, threshold)`: gộp 2 khối kề nhau nếu mean giống nhau.
3. Áp dụng split với `threshold ∈ {50, 150}`.
4. Hiển thị ảnh gốc + kết quả split (tô màu theo block).
5. Lưu kết quả vào `output/bai11_split_merge.png`.

**Lời giải:**

In [ ]:
# 1. Chuẩn bị ảnh
img = data.camera()[:256, :256]

# 2. Split bằng quadtree
def split(img, x, y, size, threshold, blocks):
    """Chia đệ quy nếu phương sai vùng > threshold."""
    block = img[y:y+size, x:x+size]
    if block.var() > threshold and size > 4:
        half = size // 2
        split(img, x,       y,       half, threshold, blocks)
        split(img, x + half, y,       half, threshold, blocks)
        split(img, x,       y + half, half, threshold, blocks)
        split(img, x + half, y + half, half, threshold, blocks)
    else:
        blocks.append((x, y, size, block.mean()))

def draw_blocks(shape, blocks):
    """Vẽ các block với giá trị trung bình."""
    out = np.zeros(shape, dtype=np.uint8)
    for (x, y, size, mean_val) in blocks:
        out[y:y+size, x:x+size] = int(mean_val)
    return out

# 3. Áp dụng với 2 ngưỡng
for T in [50, 150]:
    blocks = []
    split(img, 0, 0, 256, T, blocks)
    result = draw_blocks(img.shape, blocks)
    print(f"T={T}: số block = {len(blocks)}")

    show_grid(
        [img, result],
        ['Ảnh gốc', f'Split & Merge\nT={T}, {len(blocks)} blocks'],
        ncols=2, figsize=(11, 5),
        save_path=f'output/bai11_split_merge_T{T}.png',
        main_title=f'Bài 11: Split & Merge (ngưỡng phương sai = {T})'
    )

**📸 Kết quả:** Ngưỡng phương sai nhỏ → nhiều block nhỏ (chia mịn); ngưỡng lớn → ít block (chia thô).

---

## Bài 12: Phân cụm K-Means cho phân đoạn ảnh

**Mức độ:** Nâng cao

**Mục tiêu:** Áp dụng K-Means để phân đoạn ảnh màu.

**Yêu cầu:**
1. Tải ảnh `data.astronaut()`.
2. Chuyển sang `float32` và reshape thành `(N_pixels, 3)`.
3. Áp dụng `cv2.kmeans` với `K ∈ {2, 3, 5, 8}`.
4. Với mỗi K, hiển thị ảnh phân đoạn màu.
5. Lưu kết quả vào `output/bai12_kmeans.png`.

**Lời giải:**

In [ ]:
img_rgb = data.astronaut()

# 1. Chuẩn bị dữ liệu
pixel_values = img_rgb.reshape((-1, 3)).astype(np.float32)

# 2. Tiêu chí dừng
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.2)

# 3. Chạy K-means với nhiều K
results = []
for K in [2, 3, 5, 8]:
    _, labels, centers = cv2.kmeans(pixel_values, K, None,
                                    criteria, 10, cv2.KMEANS_RANDOM_CENTERS)
    centers = np.uint8(centers)
    segmented = centers[labels.flatten()].reshape(img_rgb.shape)
    results.append(segmented)
    print(f"K={K}: đã phân đoạn")

show_grid(
    [img_rgb] + results,
    ['Ảnh gốc'] + [f'K-Means K={K}' for K in [2, 3, 5, 8]],
    ncols=3, figsize=(15, 10),
    save_path='output/bai12_kmeans.png',
    main_title='Bài 12: Phân đoạn ảnh bằng K-Means'
)

# NHẬN XÉT:
# - K nhỏ → ảnh bị "poster hóa" mạnh, ít màu.
# - K lớn → giữ được nhiều chi tiết màu hơn.

**📸 Kết quả:** K càng lớn càng giữ được nhiều sắc thái màu.

---

## Bài 13: Superpixel với SLIC

**Mức độ:** Nâng cao

**Mục tiêu:** Tạo superpixel và khảo sát ảnh hưởng của `n_segments` và `compactness`.

**Yêu cầu:**
1. Tải ảnh `data.astronaut()`.
2. Áp dụng SLIC với các cặp:
   - `(n_segments=100, compactness=10)`
   - `(n_segments=100, compactness=30)`
   - `(n_segments=500, compactness=10)`
   - `(n_segments=500, compactness=30)`
3. Hiển thị 4 kết quả (dùng `mark_boundaries` để vẽ viền superpixel).
4. Lưu kết quả vào `output/bai13_slic.png`.

**Lời giải:**

In [ ]:
img_rgb = data.astronaut()

configs = [
    (100, 10),
    (100, 30),
    (500, 10),
    (500, 30),
]

results = []
for n_seg, comp in configs:
    segments = slic(img_rgb, n_segments=n_seg, compactness=comp,
                    sigma=1, start_label=0)
    vis = mark_boundaries(img_rgb, segments, color=(1, 0, 0))
    results.append(vis)
    print(f"n_segments={n_seg}, compactness={comp} → OK")

show_grid(
    [img_rgb] + results,
    ['Ảnh gốc'] + [f'n={n}, c={c}' for n, c in configs],
    ncols=3, figsize=(15, 10),
    save_path='output/bai13_slic.png',
    main_title='Bài 13: SLIC Superpixels — ảnh hưởng của n_segments và compactness'
)

# NHẬN XÉT:
# - n_segments lớn → superpixel nhỏ hơn, bám biên tốt hơn.
# - compactness lớn → superpixel đều đặn hơn (ưu tiên hình học).
# - compactness nhỏ  → superpixel bám màu sắc hơn (có thể méo).

**📸 Kết quả:** Superpixel bám sát biên đối tượng; `compactness` điều chỉnh độ "đều" hình dạng.

---

## Bài 14: Hough Transform — Phát hiện đường thẳng

**Mức độ:** Nâng cao

**Mục tiêu:** Áp dụng Hough Transform để phát hiện đường thẳng từ ảnh biên.

**Yêu cầu:**
1. Tạo ảnh `300×300` có 3 đường thẳng rõ ràng (ngang, dọc, chéo).
2. Phát hiện biên Canny.
3. Áp dụng `cv2.HoughLines` (Hough chuẩn) và `cv2.HoughLinesP` (xác suất).
4. Vẽ các đường phát hiện được lên ảnh gốc.
5. Hiển thị ảnh gốc, ảnh biên, kết quả Hough chuẩn, kết quả HoughLinesP.
6. Lưu kết quả vào `output/bai14_hough.png`.

**Lời giải:**

In [ ]:
# 1. Tạo ảnh
img = np.zeros((300, 300), dtype=np.uint8)
cv2.line(img, (30, 80),  (270, 80),  255, 2)   # ngang
cv2.line(img, (100, 20), (100, 280), 255, 2)   # dọc
cv2.line(img, (50, 250), (250, 50),  255, 2)   # chéo

# 2. Canny
edges = cv2.Canny(img, 50, 150)

# 3. Hough chuẩn
lines_std = cv2.HoughLines(edges, 1, np.pi/180, threshold=100)
img_hough_std = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

if lines_std is not None:
    # ✅ SỬA: dùng .flatten() để tương thích cả OpenCV 4.x và 5.0
    for line in lines_std[:10]:
        rho, theta = line.flatten()[:2]
        a, b = np.cos(theta), np.sin(theta)
        x0, y0 = a * rho, b * rho
        x1, y1 = int(x0 + 1000 * -b), int(y0 + 1000 * a)
        x2, y2 = int(x0 - 1000 * -b), int(y0 - 1000 * a)
        cv2.line(img_hough_std, (x1, y1), (x2, y2), (0, 0, 255), 1)

# 4. HoughLinesP
lines_p = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=80,
                          minLineLength=60, maxLineGap=10)
img_hough_p = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

if lines_p is not None:
    # ✅ SỬA: dùng .flatten() để tương thích cả 2 phiên bản
    for line in lines_p:
        x1, y1, x2, y2 = line.flatten()[:4]
        cv2.line(img_hough_p, (x1, y1), (x2, y2), (0, 255, 0), 2)

# 5. Hiển thị
show_grid(
    [img, edges, img_hough_std, img_hough_p],
    ['Ảnh gốc (3 đường)',
     'Canny edges',
     'Hough chuẩn (ρ, θ)',
     'HoughLinesP (xác suất)'],
    ncols=4, figsize=(18, 5),
    save_path='output/bai14_hough.png',
    main_title='Bài 14: Hough Transform — phát hiện đường thẳng'
)

**📸 Kết quả:** Cả hai đều phát hiện đúng 3 đường; HoughLinesP cho kết quả gọn hơn.

---

## Bài 15: So sánh các phương pháp phát hiện biên

**Mức độ:** Nâng cao

**Mục tiêu:** Tổng hợp so sánh Sobel, LoG, Canny trên cùng một ảnh.

**Yêu cầu:**
1. Tải ảnh `data.camera()`.
2. Tính:
   - **Sobel magnitude** + ngưỡng `T=100`.
   - **LoG** (Bài 5) + ngưỡng.
   - **Canny** với `(100, 200)`.
3. Đo **số pixel biên** của mỗi phương pháp.
4. Hiển thị 4 ảnh + bảng so sánh.
5. Lưu kết quả vào `output/bai15_compare.png`.

**Lời giải:**

In [ ]:
img = data.camera()

# 1. Sobel + ngưỡng
Gx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
Gy = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
M  = np.sqrt(Gx**2 + Gy**2)
M_norm = np.clip(M / M.max() * 255, 0, 255).astype(np.uint8)
sobel_edges = (M_norm > 100).astype(np.uint8) * 255

# 2. LoG
blurred = cv2.GaussianBlur(img, (9, 9), sigmaX=1.5)
lap = cv2.Laplacian(blurred.astype(np.float32), cv2.CV_32F)
def zero_crossing(lap):
    h, w = lap.shape
    zc = np.zeros((h, w), dtype=np.uint8)
    sign = np.sign(lap)
    for i in range(1, h-1):
        for j in range(1, w-1):
            patch = sign[i-1:i+2, j-1:j+2]
            if patch.max() > 0 and patch.min() < 0:
                zc[i, j] = 255
    return zc
log_edges = zero_crossing(lap)

# 3. Canny
canny_edges = cv2.Canny(img, 100, 200)

# 4. So sánh
methods = {
    'Sobel + T=100': sobel_edges,
    'LoG (σ=1.5)':   log_edges,
    'Canny (100,200)': canny_edges
}

print(f"{'Phương pháp':<20}{'Số pixel biên':>16}")
print("-" * 36)
for name, e in methods.items():
    print(f"{name:<20}{np.count_nonzero(e):>16,}")

show_grid(
    [img, sobel_edges, log_edges, canny_edges],
    ['Ảnh gốc'] + list(methods.keys()),
    ncols=4, figsize=(18, 5),
    save_path='output/bai15_compare.png',
    main_title='Bài 15: So sánh Sobel / LoG / Canny'
)

# NHẬN XÉT:
# - Sobel: biên dày, nhạy nhiễu.
# - LoG: biên mảnh, đôi khi có pixel cô lập.
# - Canny: biên mảnh, liên tục, ít nhiễu → tốt nhất về chất lượng.

**📸 Kết quả:** Canny cho biên mảnh và sạch nhất; Sobel cho biên dày; LoG nằm giữa.

---

## Bài 16: Pipeline phân đoạn hoàn chỉnh

**Mức độ:** Nâng cao

**Mục tiêu:** Kết hợp nhiều kỹ thuật để phân đoạn đối tượng trên nền phức tạp.

**Yêu cầu:**
1. Tải ảnh `data.coins()`.
2. Xây dựng pipeline:
   - **Bước 1:** Làm mịn Gaussian (giảm nhiễu).
   - **Bước 2:** Phân ngưỡng Otsu.
   - **Bước 3:** Morphology (mở rồi đóng) để làm sạch mask.
   - **Bước 4:** Tìm contour và vẽ lên ảnh gốc.
3. Hiển thị từng bước; đếm số đồng xu phát hiện được.
4. Lưu kết quả vào `output/bai16_pipeline.png`.

**Lời giải:**

In [ ]:
img = data.coins()

# Bước 1: Làm mịn
blur = cv2.GaussianBlur(img, (5, 5), 1.0)

# Bước 2: Otsu
_, otsu = cv2.threshold(blur, 0, 255,
                        cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# Bước 3: Morphology
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
opened = cv2.morphologyEx(otsu, cv2.MORPH_OPEN, kernel, iterations=2)
closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel, iterations=2)

# Bước 4: Contour
contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL,
                                cv2.CHAIN_APPROX_SIMPLE)
# Lọc theo diện tích
min_area = 100
valid_contours = [c for c in contours if cv2.contourArea(c) > min_area]

img_out = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
cv2.drawContours(img_out, valid_contours, -1, (0, 0, 255), 2)

print(f"Tổng số contour tìm được : {len(contours)}")
print(f"Số contour hợp lệ (>100px): {len(valid_contours)}")

show_grid(
    [img, blur, otsu, opened, closed, img_out],
    ['Ảnh gốc',
     'Gaussian blur',
     'Otsu',
     'Morphology open',
     'Morphology close',
     f'Contour ({len(valid_contours)} đồng xu)'],
    ncols=3, figsize=(15, 10),
    save_path='output/bai16_pipeline.png',
    main_title='Bài 16: Pipeline phân đoạn — Blur → Otsu → Morphology → Contour'
)

cv2.imwrite('output/bai16_mask.png', closed)
cv2.imwrite('output/bai16_contours.png', img_out)

**📸 Kết quả:** Pipeline phát hiện đúng các đồng xu, loại bỏ nền và nhiễu nhỏ.

---

## 📌 Tổng kết kiến thức được sử dụng

| Bài | Kiến thức Chương 1 → 4 |
|-----|------------------------|
| 1 | Phát hiện điểm biệt lập (mặt nạ Laplacian 8-láng giềng) |
| 2 | Phát hiện đường (4 mặt nạ định hướng) |
| 3 | Toán tử gradient: Roberts, Prewitt, Sobel |
| 4 | Canny Edge Detection |
| 5 | Marr-Hildreth (LoG): Gaussian + Laplacian + zero-crossing |
| 6 | Gradient + phân ngưỡng |
| 7 | Phân ngưỡng toàn cục: thủ công + Otsu |
| 8 | Phân ngưỡng thích nghi (Adaptive) |
| 9 | Đa ngưỡng (Multi-Otsu) |
| 10 | Phát triển vùng (Region Growing) |
| 11 | Chia tách & hợp nhất (Split & Merge / quadtree) |
| 12 | Phân cụm K-Means |
| 13 | Superpixel SLIC |
| 14 | Hough Transform — phát hiện đường thẳng |
| 15 | So sánh các phương pháp phát hiện biên |
| 16 | Pipeline phân đoạn hoàn chỉnh |

**✅ Đặc điểm:**
- Mỗi bài đều có **hiển thị ảnh minh họa** để sinh viên dễ theo dõi.
- Tất cả kết quả đều **tự động lưu** vào `output/`.
- Chỉ sử dụng kiến thức **Chương 1 → 4**; không dùng CNN, deep learning, hay mô hình học máy (thuộc Chương 5).
- Có kết hợp kiến thức Chương 1 (mask, phép toán logic, bit-plane), Chương 2 (Gaussian blur, phân ngưỡng histogram), Chương 3 (đánh giá khách quan).

**💡 Lưu ý về NumPy 2.x:** Khi làm việc với ảnh `uint8` và cần cộng/trừ có thể âm (như trong `cv2.filter2D` hay tính gradient), luôn ép về `float32`/`float64` trước, rồi mới `clip` và `astype(np.uint8)`.

Bạn có muốn tôi điều chỉnh gì trước khi gửi **Chương 5** không? 🚀